# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is published in [FAIR² - SenScience](https://doi.org/10.71728/senscience.vq4a-28xa) and described following the Croissant schema:

**URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading
Load and inspect the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display top-level metadata attributes
print(f"Dataset title: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print("\nDescription:")
print(dataset.metadata.description)


## 2. Data Overview
List available record sets, their `@id`s, and the fields/columns they provide.

In the Croissant schema, each record set and field/column is uniquely identified by its `@id`. We'll display all record set `@id`s and their content:

In [ ]:
# Gather the available record sets from the Croissant metadata
record_sets = dataset.metadata.recordSets
print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. RecordSet name: {rs.name}\n   @id: {rs['@id']}")
    # List columns for each record set
    column_ids = [col['@id'] for col in getattr(rs, 'columns', [])]
    print(f"   Columns (@id): {column_ids if column_ids else 'N/A'}\n")

### Example: Preview records (first 2) for each record set
*You may want to select a specific record set for further extraction. Here, we'll print two records from each to understand their structure.*

In [ ]:
# Print two records from each record set, referenced by their @id
for rs in dataset.metadata.recordSets:
    print(f"\nRecord set '{rs.name}' (@id: {rs['@id']}): Sample records:")
    for idx, record in enumerate(dataset.records(record_set=rs['@id'])):
        pprint.pprint(record)
        if idx >= 1:  # print only first two
            break

## 3. Data Extraction
Extract the records from the chosen record set(s) using their `@id`. Multiple dataframes will be generated if multiple record sets are present.

If you're unsure, inspect the previous overview cell for valid values. Let's extract all available sets.

In [ ]:
# List all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
dataframes = {}

for rs_id in record_set_ids:
    # records() yields dictionaries for each record
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df
    print(f"Loaded record set '{rs_id}' with shape: {df.shape}")
    print(f"Columns: {list(df.columns) if not df.empty else 'No data.'}")

# For demonstration, pick the first record set (edit as required for your analysis)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id is not None:
    print(f"\nFirst 5 rows from record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Process and explore the data: filter, normalize a numeric field, and optionally group by categorical identifiers. **All fields are referenced by their `@id`.**

In [ ]:
# Select a numeric field for analysis using its @id

if main_rs_id is not None:
    df = dataframes[main_rs_id]
    print(f"Columns available in '{main_rs_id}':\n{df.columns.tolist()}")
    
    # Let's heuristically pick a numeric column using well-known proteomics fields
    # If unsure, manually inspect and set these variables accordingly
    possible_numeric = [col for col in df.columns if col.lower().find('abundance')!=-1 or col.lower().find('count')!=-1 or col.lower().find('mw')!=-1 or col.lower().find('coverage')!=-1 or col.lower().find('intensity')!=-1]
    if len(possible_numeric) == 0:
        numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns)>0 else None
    else:
        numeric_field = possible_numeric[0]
    print(f"Using numeric field (by @id or attribute): {numeric_field}")
    
    threshold = 1000  # adjust as suitable for the dataset, e.g., abundance/count threshold
    if numeric_field is not None and numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)}/{len(df)} records with '{numeric_field}' > {threshold}")
        
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst 5 rows (with normalization):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Attempt to pick a group field (e.g., sample_id, accession, protein_name etc.)
        possible_group = [col for col in df.columns if col.lower().find('sample')!=-1 or col.lower().find('group')!=-1 or col.lower().find('desc')!=-1]
        group_field = possible_group[0] if len(possible_group)>0 else None
        print(f"Grouping by field: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
            print("\nGrouped mean by {0}:".format(group_field))
            display(grouped_df.head())
    else:
        print("No suitable numeric field detected for EDA.")


## 5. Visualization
Visualize data distributions or trends from the filtered dataset. Replace field references as needed for your context.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' in {main_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If grouped_df exists, plot group means if appropriate
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        grouped_df.sort_values(by='mean_'+numeric_field, ascending=False)[:15].plot(kind='bar', legend=False)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Top groups by mean {numeric_field}")
        plt.show()

## 6. Conclusion

- The dataset was programmatically explored using `mlcroissant`.
- We loaded records, previewed sample entries, and processed the main numeric attribute(s) for basic statistics and visualization.
- This workflow can be adapted for deeper protein abundance or modification analyses, leveraging Croissant's metadata awareness.

Further steps: experiment with additional fields, multi-set joins, or validation vs. external reference data as appropriate for your use-case.